# Lesson 3.4 — Representation vs Kinematic Singularity
**Module 6 · Unit 3 · Lesson 12**

Two unrelated failures: $\det B(\boldsymbol\phi)=0$ (representation; the angles break) vs $\operatorname{rank}J(\mathbf q)$ dropping (kinematic; the robot loses motion). We show gimbal lock leaves $J$ full rank, and a folded arm drops $\operatorname{rank}J$ while $B$ stays fine.

In [1]:
import numpy as np
def skew(v):
    v=np.asarray(v,float).ravel(); return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i])
        M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def pose(P,T,q):
    M=forward_chain(P,T,q)[-1]; return M[:3,3], M[:3,:3]

# ZYX roll-pitch-yaw
def rpy_from_R(R):
    return np.array([np.arctan2(R[2,1],R[2,2]),
                     np.arctan2(-R[2,0],np.hypot(R[2,1],R[2,2])),
                     np.arctan2(R[1,0],R[0,0])])   # [roll, pitch, yaw]
def R_from_rpy(rpy):
    r,p,y=rpy
    Rz=np.array([[np.cos(y),-np.sin(y),0],[np.sin(y),np.cos(y),0],[0,0,1.0]])
    Ry=np.array([[np.cos(p),0,np.sin(p)],[0,1,0],[-np.sin(p),0,np.cos(p)]])
    Rx=np.array([[1,0,0],[0,np.cos(r),-np.sin(r)],[0,np.sin(r),np.cos(r)]])
    return Rz@Ry@Rx

# spatial 3R test arm
ARM=([(0,0,0,np.pi/2),(0,0,1.0,0),(0,0,0.8,0)],["R","R","R"])


## At gimbal lock: det B → 0 but the geometric J is full rank (robot is fine)

In [2]:
checks=[]
def Bmat(rpy,eps=1e-7):
    R=R_from_rpy(rpy); B=np.zeros((3,3))
    for k in range(3):
        e=np.zeros(3); e[k]=eps; dR=(R_from_rpy(rpy+e)-R_from_rpy(rpy-e))/(2*eps); B[:,k]=vee(dR@R.T)
    return B
P,T=ARM; q=np.array([0.3,0.5,-0.4])
detB=np.linalg.det(Bmat(np.array([0.2,np.pi/2-1e-6,0.5])))   # representation at gimbal
rankJ=np.linalg.matrix_rank(geometric_jacobian(P,T,q),tol=1e-6)
print("det B near gimbal =",round(detB,6),"  rank J (arm) =",rankJ,"of",len(q))
checks.append(abs(detB)<1e-2)         # representation breaks
checks.append(rankJ==len(q))          # robot fine

det B near gimbal = 1e-06   rank J (arm) = 3 of 3


## A folded planar 2R arm: geometric J drops rank (kinematic), B unrelated

In [3]:
P2=[(0,0,1.0,0),(0,0,1.0,0)]; T2=["R","R"]
q_sing=np.array([0.3,0.0])            # theta2 = 0 -> straight arm -> singular
J2=geometric_jacobian(P2,T2,q_sing)[:2,:]    # planar: x,y rows
print("planar J at straight pose:\n",np.round(J2,3))
print("rank =",np.linalg.matrix_rank(J2,tol=1e-6),"(of 2) -> kinematic singularity")
checks.append(np.linalg.matrix_rank(J2,tol=1e-6)<2)
# and a benign orientation keeps B well-conditioned
checks.append(abs(np.linalg.det(Bmat(np.array([0.1,0.1,0.1]))))>0.5)

planar J at straight pose:
 [[-0.591 -0.296]
 [ 1.911  0.955]]
rank = 1 (of 2) -> kinematic singularity


## Summary: which matrix diagnoses which failure

In [4]:
print("representation singularity  -> inspect det B(phi)")
print("kinematic singularity       -> inspect rank / singular values of J(q)")
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

representation singularity  -> inspect det B(phi)
kinematic singularity       -> inspect rank / singular values of J(q)
All checks passed.
